# Lab: Tree models, neural networks, and SGD

**Due: Monday 7 December 2026, the last day of classes**

**How to submit on Canvas.** Restart the kernel and Run All. Every check cell must print `ok`. Save the notebook with its outputs and upload the `.ipynb` file to the Canvas assignment for this lab.

**What a check says.** `ok  name` means your value matches the right one (to 6 decimals, or within the stated tolerance). An error that starts with `not yet` means the value is wrong or is still `None`. Fix the exercise cell and run it again. A check never shows the right value.

**Data.** The first cell downloads `diamonds.csv`, `credit.csv`, `circle.csv` and `SaratogaHouses.csv` into `data/`. Allowed libraries: `numpy`, `pandas`, `scikit-learn`, `torch`.

| Exercise | Task | Slide topic |
|---|---|---|
| 1 | Best split by SSE | Regression trees: choosing a split |
| 2 | Gini index of a split | Classification trees: impurity |
| 3 | Pruning strength chosen on validation data | Overfitting, pruning, validation |
| 4 | Random forest and boosting scores | Bagging, random forests, boosting |
| 5 | Forward pass of a small network | Neural networks: hidden layer, ReLU |
| 6 | Softmax, cross-entropy and its gradient | Multi-class output and loss |
| 7 | Gradient descent steps | Gradient descent |
| 8 | Minibatch SGD in PyTorch | SGD, training loop, checkpoint |


In [ ]:
import copy
from hashlib import sha256
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DATA = Path("data")
DATA.mkdir(exist_ok=True)
BASE = "https://vsokolov.org/courses/files/600/data"
for name in ("diamonds.csv", "credit.csv", "circle.csv", "SaratogaHouses.csv"):
    if not (DATA / name).exists():
        urlretrieve(f"{BASE}/{name}", DATA / name)

PREFIX = "aii600-trees-nn-sgd"


def check(name, value, expected, ndigits=6):
    if value is None:
        raise AssertionError(f"not yet: {name} is still None")
    try:
        x = float(value)
    except (TypeError, ValueError):
        raise AssertionError(f"not yet: {name} must be a plain number, not {type(value).__name__}")
    s = f"{round(x, ndigits):.{ndigits}f}"
    got = sha256(f"{PREFIX}|{name}|{s}".encode()).hexdigest()[:16]
    assert got == expected, f"not yet: {name} is not right (yours: {x:.6g})"
    print("ok ", name)


def check_array(name, value, expected, shape, ndigits=6):
    if value is None:
        raise AssertionError(f"not yet: {name} is still None")
    arr = np.asarray(value, dtype=float)
    if arr.shape != tuple(shape):
        raise AssertionError(f"not yet: {name} has shape {arr.shape}, the task asks for {tuple(shape)}")
    key = (np.round(arr, ndigits) + 0.0).tobytes()
    got = sha256(f"{PREFIX}|{name}|".encode() + key).hexdigest()[:16]
    assert got == expected, f"not yet: {name} has the right shape but some values are wrong"
    print("ok ", name)


def check_near(name, value, target, tol, relative=False):
    # for library results that can differ a little between versions
    if value is None:
        raise AssertionError(f"not yet: {name} is still None")
    gap = abs(float(value) - target) / (abs(target) if relative else 1.0)
    assert gap < tol, f"not yet: {name} is not within the tolerance of the right value (yours: {float(value):.6g})"
    print("ok ", name)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))


### Exercise 1: best split by SSE

**Task.** Find the best split of the 8 diamonds in `small`. Candidate cuts are the midpoints between neighbouring distinct carat values. Diamonds with carat below the cut go left, the others go right.

**Inputs.** `small` (DataFrame with `carat` in carats and `price` in US dollars) and `sse(values)` (sum of squared deviations from the mean).

**Formula.** score(cut) = SSE(left prices) + SSE(right prices). Keep the cut with the smallest score.

**Define.** `best_cut` (carats) and `best_sse` (squared dollars). Do not round.


In [ ]:
small = pd.DataFrame([
    (0.4, 855),
    (0.99, 5893),
    (0.51, 1839),
    (0.7, 1066),
    (1.0, 4042),
    (1.0, 5544),
    (0.82, 2165),
    (0.7, 2657)
], columns=["carat", "price"])

def sse(values):
    values = np.asarray(values, float)
    return float(((values - values.mean()) ** 2).sum())

print(small.T.to_string())


In [ ]:
# Exercise 1: define best_cut, best_sse
best_cut = None
best_sse = None


In [ ]:
check("best_cut", best_cut, "a9f6470127ee27fc")
check("best_sse", best_sse, "2562b2dc05cc1c3a")


### Exercise 2: Gini index of a split

**Task.** Score the split "loan duration is more than 24 months" by weighted Gini.

**Inputs.** `credit` (columns `Default`, 1 for default and 0 otherwise, and `duration` in months).

**Formula.** For a group with default share $p$, Gini $= 1 - p^2 - (1-p)^2$. After the split, `gini_after` $= \frac{n_L}{n} G_L + \frac{n_R}{n} G_R$, where the left group L has duration above 24, the right group R has duration at most 24, and $n$ is the number of loans.

**Define.** `gini_parent` (Gini of all loans) and `gini_after`. Do not round.


In [ ]:
credit = pd.read_csv(DATA / "credit.csv")
print(len(credit), "loans; default share", round(credit.Default.mean(), 3))


In [ ]:
# Exercise 2: define gini_parent, gini_after
gini_parent = None
gini_after = None


In [ ]:
check("gini_parent", gini_parent, "bc6fadcb34f27ec6")
check("gini_after", gini_after, "52129127a5de6186")


### Exercise 3: pruning strength chosen on validation data

**Task.** For each `alpha` in `alphas`, fit `DecisionTreeRegressor(random_state=0, ccp_alpha=alpha)` on `Xtr, ytr` and compute the validation RMSE on `Xva, yva` with `rmse`. Keep the alpha with the smallest validation RMSE (the first one if two tie).

**Inputs.** `Xtr, ytr, Xva, yva` (Saratoga homes, 70% training and 30% validation, `price` in US dollars), `alphas`, `rmse`.

**Define.** `best_val_rmse` (US dollars) and `best_leaves` (integer, leaves of the winning tree).

**Tolerance.** `best_val_rmse` within 0.5% and `best_leaves` within 3 of the right values (scikit-learn versions differ a little).


In [ ]:
homes = pd.read_csv(DATA / "SaratogaHouses.csv", index_col=0)
Xall = pd.get_dummies(homes.drop(columns="price"), drop_first=True).astype(float)
yall = homes.price.to_numpy()
order = np.random.default_rng(10).permutation(len(homes))
n_train = int(0.7 * len(homes))
Xtr, ytr = Xall.iloc[order[:n_train]], yall[order[:n_train]]
Xva, yva = Xall.iloc[order[n_train:]], yall[order[n_train:]]
alphas = DecisionTreeRegressor(random_state=0).cost_complexity_pruning_path(Xtr, ytr).ccp_alphas[::10]
print(len(Xtr), "training rows,", len(Xva), "validation rows,", len(alphas), "alphas")


In [ ]:
# Exercise 3: define best_val_rmse, best_leaves
best_val_rmse = None
best_leaves = None


In [ ]:
check_near("best_val_rmse", best_val_rmse, 60191.31, 0.005, relative=True)
check_near("best_leaves", best_leaves, 11, 3.5)


### Exercise 4: random forest and boosting scores

**Task.** Fit both models on `Xtr, ytr` from Exercise 3 and score them. Use exactly these settings and seeds.

- Forest: `RandomForestRegressor(n_estimators=200, max_features=1/3, min_samples_leaf=3, oob_score=True, random_state=0, n_jobs=-1)`. Its out-of-bag R² is the attribute `oob_score_`.
- Boosting: `GradientBoostingRegressor(n_estimators=300, learning_rate=0.1, max_depth=3, random_state=0)`. Score it on `Xva, yva` with `rmse`.

**Define.** `oob_r2` (a number below 1, no unit) and `gb_val_rmse` (US dollars).

**Tolerance.** `oob_r2` within 0.02 and `gb_val_rmse` within 2% of the right values.


In [ ]:
# inputs: Xtr, ytr, Xva, yva, rmse from Exercise 3


In [ ]:
# Exercise 4: define oob_r2, gb_val_rmse
oob_r2 = None
gb_val_rmse = None


In [ ]:
check_near("oob_r2", oob_r2, 0.6525, 0.02)
check_near("gb_val_rmse", gb_val_rmse, 50751.4, 0.02, relative=True)


### Exercise 5: forward pass of a small network

**Task.** Run the forward pass of the network below for one new point and for all 200 circle points.

**Inputs.** `W1` (shape (2, 4)), `b1` (4,), `W2` (4,), `b2` (number), `x_new` (2,), `Xcirc` (200, 2), `ycirc` (200 labels, 0 for the center and 1 for the outer ring). There is no `relu` function: use `np.maximum(..., 0.0)`.

**Formula.** Hidden values $h = \max(x W_1 + b_1,\, 0)$ and output $o = h \cdot W_2 + b_2$. Predict class 1 when $o > 0$, otherwise class 0.

**Define.**
- `h_new`: hidden values for `x_new`, shape (4,).
- `y_new`: the output $o$ for `x_new`, a number.
- `H_all`: hidden values for all points, shape (200, 4).
- `acc_circle`: share of the 200 points whose predicted class equals `ycirc`, between 0 and 1.

Do not round.


In [ ]:
circle = pd.read_csv(DATA / "circle.csv")
Xcirc = circle[["x1", "x2"]].to_numpy()
ycirc = circle.label.to_numpy()
W1 = np.array([[1.0, -1.0, 0.0, 0.0],
               [0.0, 0.0, 1.0, -1.0]])
b1 = np.array([-4.0, -4.0, -4.0, -4.0])
W2 = np.array([1.0, 1.0, 1.0, 1.0])
b2 = -0.5
x_new = np.array([-6.0, 5.5])
print(Xcirc.shape, ycirc.shape)


In [ ]:
# Exercise 5: define h_new, y_new, H_all, acc_circle
h_new = None
y_new = None
H_all = None
acc_circle = None


In [ ]:
check_array("h_new", h_new, "4771e4785f1e4c5c", (4,))
check("y_new", y_new, "a98b80685db68ac1")
check_array("H_all", H_all, "9512dd8cd2b7e589", (200, 4))
check("acc_circle", acc_circle, "bbaa670b89efc325")


### Exercise 6: softmax, cross-entropy and its gradient

**Task.** For 4 examples and 3 classes, compute the probabilities, the mean loss and the gradient of the mean loss with respect to the scores. Work along the class axis (`axis=1`) and subtract each row's maximum before `np.exp`.

**Inputs.** `S_ex` (shape (4, 3), raw scores) and `lab_ex` (4 labels, each 0, 1 or 2).

**Formula.** $p_{ik} = e^{s_{ik}} / \sum_j e^{s_{ij}}$. Loss $= -\frac{1}{n}\sum_i \log p_{i,c_i}$, where $c_i$ is the true class of example $i$ and $n = 4$. Gradient $= (P - Y)/n$, where $Y$ is the one-hot matrix `np.eye(3)[lab_ex]`.

**Define.** `P_ex` (shape (4, 3)), `loss_ex` (a number, in nats) and `grad_scores` (shape (4, 3)). Do not round.


In [ ]:
S_ex = np.array([[2.0, 0.5, -1.0], [0.0, 0.0, 0.0], [1.0, 3.0, 0.0], [-1.0, 0.0, 2.0]])
lab_ex = np.array([0, 2, 1, 2])


In [ ]:
# Exercise 6: define P_ex, loss_ex, grad_scores
P_ex = None
loss_ex = None
grad_scores = None


In [ ]:
check_array("P_ex", P_ex, "b95c673a8f668257", (4, 3))
check("loss_ex", loss_ex, "1ade89f06c21799a")
check_array("grad_scores", grad_scores, "e31f99c3062f1e7a", (4, 3))


### Exercise 7: gradient descent steps

**Task.** Fit the one-weight logistic model $p_i = \sigma(w x_i)$ by gradient descent. First do one step with the formula, then five steps in a loop.

**Inputs.** `x_ex`, `y_ex` (arrays of length 4), `w_start_ex` (starting weight), `eta_ex` (learning rate), `sigmoid`.

**Formulas.** Gradient of the mean log loss: $g(w) = \frac{1}{4}\sum_i (p_i - y_i)\,x_i$. Update: $w \leftarrow w - \eta\, g(w)$. Mean log loss: $L(w) = -\frac{1}{4}\sum_i [\,y_i \log p_i + (1-y_i)\log(1-p_i)\,]$.

**Define.**
- `grad_ex`: $g$ at `w_start_ex`.
- `w_new`: the weight after one step from `w_start_ex`.
- `w_after5`: the weight after five steps in total, starting from `w_start_ex`.
- `loss_after5`: $L$ at `w_after5`.

Do not round.


In [ ]:
x_ex = np.array([1.0, -2.0, 0.5, 3.0])
y_ex = np.array([1.0, 0.0, 0.0, 1.0])
w_start_ex, eta_ex = 0.5, 0.4


In [ ]:
# Exercise 7: define grad_ex, w_new, w_after5, loss_after5
grad_ex = None
w_new = None
w_after5 = None
loss_after5 = None


In [ ]:
check("grad_ex", grad_ex, "37e8416a26ab696f")
check("w_new", w_new, "7f7d619c6c8a4148")
check("w_after5", w_after5, "8e44ebffe61677dc")
check("loss_after5", loss_after5, "d1f9a53b46a98912")


### Exercise 8: minibatch SGD in PyTorch

**Task.** Train a network on the simulated data with plain SGD in minibatches of 64, for 5 epochs, with the fixed seeds already in the cell. After each epoch record the validation loss. Keep a copy of the weights with the lowest validation loss and restore them at the end.

**Inputs.** `make_loader(0)` (training batches), `make_model(0)` (784 to 32 to 10 network), `evaluate(model, loader)` (returns average cross-entropy and accuracy), `val_loader`, learning rate 0.2.

**Define.** `val_loss` (cross-entropy in nats) and `val_acc` (between 0 and 1) of the restored model on the validation set.

**Tolerance.** `val_loss` within 0.1 and `val_acc` within 0.06 of the right values (other seeds move the result by up to 0.08 and 0.05). The check also tests `best_epoch` and the restore.


In [ ]:
# SIMULATED data: 10 classes, 784 features. Each class has a random centre; an example is its centre (scaled) plus noise.
def synthetic_split(n_train=2000, n_val=1000):
    r = np.random.default_rng(0)
    centres = r.normal(size=(10, 784))
    def draw(n):
        y = r.integers(0, 10, size=n)
        x = 0.1 * centres[y] + r.normal(size=(n, 784))
        return TensorDataset(torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return draw(n_train), draw(n_val)

train_set, val_set = synthetic_split()
val_loader = DataLoader(val_set, batch_size=1000)

def make_loader(seed=0):
    return DataLoader(train_set, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(seed))

def make_model(seed=0):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(784, 32), nn.ReLU(), nn.Linear(32, 10))

def evaluate(model, loader):
    # average cross-entropy and accuracy, without changing the weights
    model.eval()
    with torch.no_grad():
        xb_, yb_ = next(iter(loader))
        out = model(xb_)
        result = nn.functional.cross_entropy(out, yb_).item(), (out.argmax(dim=1) == yb_).float().mean().item()
    model.train()
    return result
print(len(train_set), "training examples,", len(val_set), "validation examples (simulated data)")


In [ ]:
# Exercise 8: fill in each line marked FILL, then define val_loss and val_acc
train_loader = make_loader(0)
model = make_model(0)
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.2)
val_curve, best_val, best_epoch, best_state = [], float("inf"), None, None
for epoch in range(5):
    for xb, yb in train_loader:
        scores = None   # FILL: forward pass (raw scores)
        loss = None     # FILL: loss from scores and yb
        # FILL: zero the gradients
        # FILL: backward pass
        # FILL: SGD step
    val_curve.append(evaluate(model, val_loader)[0])
    # FILL: if this epoch has the lowest validation loss so far, set best_val, best_epoch (from 0) and best_state (a copy.deepcopy of model.state_dict())
# FILL: load best_state into model
val_loss = None   # FILL: validation loss of the restored model
val_acc = None    # FILL: validation accuracy of the restored model


In [ ]:
assert best_epoch is not None and best_epoch == int(np.argmin(val_curve)), "not yet: best_epoch must be the epoch with the lowest validation loss"
assert abs(evaluate(model, val_loader)[0] - min(val_curve)) < 1e-5, "not yet: restore the best weights into model"
check_near("val_loss", val_loss, 0.9202, 0.1)
check_near("val_acc", val_acc, 0.6990, 0.06)
